# Data Preparation — Smart Community ML


## 1. Imports and project paths

In [1]:
from pathlib import Path
import csv
import hashlib
import json
import re
import unicodedata

import numpy as np
import pandas as pd


In [2]:
def find_project_root():
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]

    for candidate in candidates:
        if (candidate / "src" / "prepare_data.py").is_file():
            return candidate

    expected = Path.home() / "Documents" / "Group-03_Social-Community-app"
    if (expected / "src" / "prepare_data.py").is_file():
        return expected.resolve()

    raise FileNotFoundError(
        "Could not find the project root. Start Jupyter from the "
        "Group-03_Social-Community-app directory, or update the "
        "expected project path in this cell."
    )


ROOT = find_project_root()
JIGSAW_DIR = ROOT / "data" / "raw" / "jigsaw"
SMS_PATH = ROOT / "data" / "raw" / "sms" / "SMSSpamCollection"
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Jigsaw directory:", JIGSAW_DIR)
print("SMS file:", SMS_PATH)
print("Output directory:", OUT)


Project root: /Users/kakulrajput/Documents/Social_Community_app_ML
Jigsaw directory: /Users/kakulrajput/Documents/Social_Community_app_ML/data/raw/jigsaw
SMS file: /Users/kakulrajput/Documents/Social_Community_app_ML/data/raw/sms/SMSSpamCollection
Output directory: /Users/kakulrajput/Documents/Social_Community_app_ML/data/processed


## 2. Label and output-column definitions

In [3]:
LABELS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate",
]

COLS = [
    "id",
    "text",
    "source_dataset",
    "source_row_id",
    "spam_label",
    "spam_label_available",
    *LABELS,
    "toxicity_labels_available",
]


## 3. Text normalization and column validation

In [4]:
def normalize_text(value):
    if pd.isna(value):
        return ""
    value = unicodedata.normalize("NFKC", str(value))
    value = value.replace("\u200b", "").replace("\ufeff", "")
    return re.sub(r"\s+", " ", value).strip()


def require_columns(df, columns, filename):
    missing = set(columns) - set(df.columns)
    if missing:
        raise ValueError(f"{filename} is missing columns: {sorted(missing)}")


## 4. Load and prepare the four Jigsaw files

In [5]:
def read_jigsaw_files():
    paths = {
        "train": JIGSAW_DIR / "train.csv",
        "test": JIGSAW_DIR / "test.csv",
        "labels": JIGSAW_DIR / "test_labels.csv",
        "sample_submission": JIGSAW_DIR / "sample_submission.csv",
    }

    for name, path in paths.items():
        if not path.is_file():
            raise FileNotFoundError(f"Missing Jigsaw {name} file: {path}")

    train_raw = pd.read_csv(paths["train"])
    test_raw = pd.read_csv(paths["test"])
    labels_raw = pd.read_csv(paths["labels"])
    sample_submission = pd.read_csv(paths["sample_submission"])

    require_columns(train_raw, ["id", "comment_text", *LABELS], "train.csv")
    require_columns(test_raw, ["id", "comment_text"], "test.csv")
    require_columns(labels_raw, ["id", *LABELS], "test_labels.csv")
    require_columns(
        sample_submission,
        ["id", *LABELS],
        "sample_submission.csv",
    )

    train = train_raw[["id", "comment_text", *LABELS]].rename(
        columns={"comment_text": "text"}
    )
    train["text"] = train["text"].map(normalize_text)
    train = train[train["text"].ne("")].copy()
    train = train.drop_duplicates(subset=["id"], keep="first")

    for label in LABELS:
        train[label] = pd.to_numeric(train[label], errors="coerce")

    train = train.dropna(subset=LABELS).copy()

    for label in LABELS:
        if not train[label].isin([0, 1]).all():
            raise ValueError(f"train.csv contains non-binary values in {label}")
        train[label] = train[label].astype("int8")

    train["source_dataset"] = "jigsaw_toxic_comment"
    train["source_row_id"] = train["id"].astype(str)
    train["spam_label"] = pd.NA
    train["spam_label_available"] = False
    train["toxicity_labels_available"] = True
    train = train[COLS].reset_index(drop=True)

    test = test_raw[["id", "comment_text"]].rename(
        columns={"comment_text": "text"}
    )
    test["text"] = test["text"].map(normalize_text)
    test = test[test["text"].ne("")].copy()
    test = test.drop_duplicates(subset=["id"], keep="first")
    test["source_dataset"] = "jigsaw_toxic_comment_test"
    test["source_row_id"] = test["id"].astype(str)
    test["spam_label"] = pd.NA
    test["spam_label_available"] = False

    for label in LABELS:
        test[label] = pd.NA

    test["toxicity_labels_available"] = False
    test = test[COLS].reset_index(drop=True)

    labels = labels_raw[["id", *LABELS]].copy()

    for label in LABELS:
        labels[label] = pd.to_numeric(labels[label], errors="coerce")

    labeled_test = test[
        ["id", "text", "source_dataset", "source_row_id"]
    ].merge(
        labels,
        on="id",
        how="inner",
        validate="one_to_one",
    )

    valid_mask = labeled_test[LABELS].isin([0, 1]).all(axis=1)
    labeled_test = labeled_test.loc[valid_mask].copy()

    for label in LABELS:
        labeled_test[label] = labeled_test[label].astype("int8")

    labeled_test["spam_label"] = pd.NA
    labeled_test["spam_label_available"] = False
    labeled_test["toxicity_labels_available"] = True
    labeled_test["source_dataset"] = "jigsaw_official_test_labeled"
    labeled_test = labeled_test[COLS].reset_index(drop=True)

    sample_submission.to_csv(
        OUT / "jigsaw_sample_submission.csv",
        index=False,
    )

    return train, test, labeled_test, sample_submission, len(labels_raw)


## 5. Load and prepare the SMS Spam Collection

In [6]:
def read_sms():
    if not SMS_PATH.is_file():
        raise FileNotFoundError(f"Missing SMS dataset: {SMS_PATH}")

    sms = pd.read_csv(
        SMS_PATH,
        sep="\t",
        header=None,
        names=["spam_label", "text"],
        encoding="utf-8",
        quoting=csv.QUOTE_NONE,
        on_bad_lines="error",
    )

    if not set(sms["spam_label"].dropna().unique()).issubset({"ham", "spam"}):
        raise ValueError("SMS dataset contains unexpected labels")

    sms["text"] = sms["text"].map(normalize_text)
    sms = sms[sms["text"].ne("")].drop_duplicates(
        subset=["spam_label", "text"],
        keep="first",
    ).copy()
    sms = sms.reset_index(drop=True)

    sms["id"] = pd.NA
    sms["source_dataset"] = "sms_spam_collection"
    sms["source_row_id"] = [
        hashlib.sha256(f"sms\0{label}\0{text}".encode()).hexdigest()[:20]
        for label, text in zip(sms["spam_label"], sms["text"])
    ]
    sms["spam_label_available"] = True
    sms["toxicity_labels_available"] = False

    for label in LABELS:
        sms[label] = pd.NA

    return sms[COLS].reset_index(drop=True)


## 6. Run preparation and save processed datasets

In [7]:
train, official_test, official_test_labeled, sample_submission, raw_test_label_rows = read_jigsaw_files()
sms = read_sms()

train.to_csv(OUT / "jigsaw_cleaned.csv", index=False)
official_test.to_csv(OUT / "jigsaw_official_test.csv", index=False)
official_test_labeled.to_csv(
    OUT / "jigsaw_official_test_labeled.csv",
    index=False,
)
sms.to_csv(OUT / "sms_cleaned.csv", index=False)

master = pd.concat(
    [train, official_test, sms],
    ignore_index=True,
)
master.to_csv(OUT / "unified_master.csv", index=False)

manifest = {
    "jigsaw_train_clean_rows": int(len(train)),
    "jigsaw_official_test_rows": int(len(official_test)),
    "jigsaw_test_labels_raw_rows": int(raw_test_label_rows),
    "jigsaw_official_test_labeled_rows": int(len(official_test_labeled)),
    "jigsaw_sample_submission_rows": int(len(sample_submission)),
    "sms_clean_rows": int(len(sms)),
    "unified_master_rows": int(len(master)),
    "source_counts": master["source_dataset"].value_counts().to_dict(),
    "sms_label_counts": sms["spam_label"].value_counts().to_dict(),
    "jigsaw_positive_counts": {
        label: int(train[label].sum()) for label in LABELS
    },
    "official_test_label_policy": (
        "Rows containing -1 or missing labels are excluded from "
        "the labeled evaluation output."
    ),
    "sample_submission_policy": (
        "Saved for reference only; never used as training labels."
    ),
    "cleaning": (
        "Unicode NFKC; remove zero-width spaces and BOM; collapse "
        "whitespace; trim; preserve punctuation, case, negation, and emojis."
    ),
    "label_policy": (
        "Unknown labels remain missing; spam labels are not toxicity labels."
    ),
}

(OUT / "data_manifest.json").write_text(
    json.dumps(manifest, indent=2, default=str),
    encoding="utf-8",
)

print(json.dumps(manifest, indent=2, default=str))
print(f"\nProcessed files saved to: {OUT}")


{
  "jigsaw_train_clean_rows": 159571,
  "jigsaw_official_test_rows": 153163,
  "jigsaw_test_labels_raw_rows": 153164,
  "jigsaw_official_test_labeled_rows": 63978,
  "jigsaw_sample_submission_rows": 153164,
  "sms_clean_rows": 5160,
  "unified_master_rows": 317894,
  "source_counts": {
    "jigsaw_toxic_comment": 159571,
    "jigsaw_toxic_comment_test": 153163,
    "sms_spam_collection": 5160
  },
  "sms_label_counts": {
    "ham": 4518,
    "spam": 642
  },
  "jigsaw_positive_counts": {
    "toxic": 15294,
    "severe_toxic": 1595,
    "obscene": 8449,
    "threat": 478,
    "insult": 7877,
    "identity_hate": 1405
  },
  "official_test_label_policy": "Rows containing -1 or missing labels are excluded from the labeled evaluation output.",
  "sample_submission_policy": "Saved for reference only; never used as training labels.",
  "cleaning": "Unicode NFKC; remove zero-width spaces and BOM; collapse whitespace; trim; preserve punctuation, case, negation, and emojis.",
  "label_policy"

## 7. Quick checks

These checks confirm the expected files were written and the label-availability flags are consistent.


In [8]:
expected_outputs = [
    "jigsaw_cleaned.csv",
    "jigsaw_official_test.csv",
    "jigsaw_official_test_labeled.csv",
    "jigsaw_sample_submission.csv",
    "sms_cleaned.csv",
    "unified_master.csv",
    "data_manifest.json",
]

output_check = pd.DataFrame({
    "file": expected_outputs,
    "exists": [(OUT / filename).is_file() for filename in expected_outputs],
})

display(output_check)

assert output_check["exists"].all(), "One or more expected output files are missing."
assert train["toxicity_labels_available"].all()
assert not official_test["toxicity_labels_available"].any()
assert official_test_labeled["toxicity_labels_available"].all()
assert sms["spam_label_available"].all()

print("All preparation checks passed.")


,file,exists
0,jigsaw_cleaned.csv,True
1,jigsaw_official_test.csv,True
2,jigsaw_official_test_labeled.csv,True
3,jigsaw_sample_submission.csv,True
4,sms_cleaned.csv,True
5,unified_master.csv,True
6,data_manifest.json,True


All preparation checks passed.
